In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
#Quantum Dice
#Amitava Chakraborty - 19th Dec 2019
#Issue : After around 4000 rolls, the sequence repeats. 
#So if you roll the same dice over 4000 times, the pattern repeats and thus is no longer random. 
#The circuit uses only 5 qubits, the app doesn't support a die larger than 32 sides. 

from qiskit_backend_config import IBMQ
# Loading your IBM Q account(s)
IBMQ.load_account()

In [ ]:
import numpy as np
import math
# Importing standard Qiskit libraries and configuring account
from qiskit import QuantumCircuit
from qiskit_backend_config import execute
from qiskit_backend_config import BasicAer

backend = BasicAer.get_backend('qasm_simulator')

Classical computers can, at best, generate pseudo random numbers. 
Quantum computers, using uniquely quantum properties, can generate truly random numbers. 

This example uses random numbers generated from IBM Q's quantum systems 
to provide a random sequence for different types of dice.

In [ ]:
#Circuit to obtain random numbers
#The circuit uses 5 qubits - to each a hadamard gate is added.
#The H gate sets that qubit into a superposition state of 0 and 1. 
#A measurement gate is then applied to each qubit - there is a 50% chance of getting a 0, and a 50% of getting a 1. 
#When all 5 qubits are measured, we will get a result - any 5 bit combination of 0s and 1s 
#Because all qubits had an equal chance of being 0 or 1, 
#the output had an equal chance of being anything between 00000 and 11111.


qc = QuantumCircuit(5, 5)
qc.h(0)
qc.h(1)
qc.h(2)
qc.h(3)
qc.h(4)
qc.measure([0, 1, 2, 3, 4], [0, 1, 2, 3, 4])
qc.draw('mpl')

In [ ]:
#We use Python to convert those binary numbers to decimals from 0 to 31 
#Hence this circuit is a random number generator for values between 0 and 31. 

#IBMQs are not yet perfect - 
#so the probalibility distribution to generate true random numbers is close to a uniform distribution, but not fully uniform
#Imperfections are due to sources of noise--such as errors in measurement--
#and some of it is due to the number of times we rolled the dice. 
#The lower the noise and the more rolls we make, the closer the observed distribution will be to uniform.

#We ran the above circuit several thousand times and stored the results.
#The sequence is truly random unless we do more than 4000 dice rolls. 

shots = 4096
job = execute(qc, backend, shots=shots, memory=True)

In [ ]:
#we want to see results independently, instead of as a probability
result = job.result()
memory = result.get_memory()

# need an array to drop all the results into
outputArray = []

#convert results to decimal and drop into array
for i in range(0, shots):
  converted = int(memory[i], 2)
  outputArray.append(converted)

print(outputArray)
#That's it for getting a truly random sequence from our quantum system. 

In [ ]:
#Next, let's look at building a simple web app around the results.
import random
index = math.floor(random.random() * len(outputArray))

#When someone rolls a dice, we need to normalize our or array of random numbers. 
#Technically speaking, its the sequence that is random, not the numbers. 
#Once normalized, we'll multiply the value by the size of the dice and round up 
#(most dice don't include decimal places. Ours won't either). 
#We also add a "plus 1" because we don't want our dice to roll a zero.

#maxDiceRoll is 32, because 5 bits has a max of 32.
maxDiceRoll=32
#How many faces the dice has
diceSize=6
normalizedRoll = (outputArray[index] + 1)/maxDiceRoll
print(outputArray[index],normalizedRoll)
result = math.ceil(normalizedRoll * diceSize);
print(result)